# SolveIt Slash Commands

Type `/command args` in any code cell to run a slash command.
Commands live in `commands/*.md`. Create new ones with `/new_command`.

Available: `/compact`, `/repro`, `/tidy`, `/simplify`, `/craft`, `/new_command`

In [ ]:
import re, subprocess
from pathlib import Path
from IPython import get_ipython
from dialoghelper import *

async def slash(name: str, args: str = ""):
    "Load commands/{name}.md, expand variables, add as prompt"
    cmd_dir = Path('/app/data/commands')
    cmd_file = cmd_dir / f'{name}.md'
    if not cmd_file.exists():
        avail = [f.stem for f in cmd_dir.glob('*.md') if f.stem != 'README']
        print(f'❌  No command: {name}')
        print(f'   Available: {", ".join(avail)}')
        return
    tmpl = cmd_file.read_text()
    if tmpl.startswith('---\n'):  # strip YAML frontmatter
        parts = tmpl.split('---\n', 2)
        if len(parts) == 3: tmpl = parts[2].strip()
    tmpl = tmpl.replace('$ARGUMENTS', args)
    for i, a in enumerate(args.split(), 1): tmpl = tmpl.replace(f'${i}', a)
    def _file(m):
        p = Path(m.group(1)).expanduser()
        return p.read_text() if p.exists() else m.group(0)
    tmpl = re.sub(r'@(\S+)', _file, tmpl)
    def _sh(m): return subprocess.run(m.group(1), shell=True, capture_output=True, text=True).stdout.strip()
    tmpl = re.sub(r'!`([^`]+)`', _sh, tmpl)
    await add_msg(tmpl, type='prompt')
    print(f'✓ /{name} → prompt added')

def _slash_transform(lines):
    if lines and re.match(r'\s*/\w', lines[0]):
        parts = lines[0].strip().lstrip('/').split(None, 1)
        lines[0] = f"await slash({parts[0]!r}, {(parts[1] if len(parts)>1 else '')!r})\n"
    return lines

get_ipython().input_transformers_cleanup.append(_slash_transform)
Path('/app/data/commands').mkdir(exist_ok=True)
print('✓ Slash commands ready — try /new_command <name> <description>')